# 悬臂梁问题

**类别：** 仿真

来源：[https://www.hexaly.com/templates/cantilevered-beam-problem](https://www.hexaly.com/templates/cantilevered-beam-problem)


## 问题描述

**悬臂梁问题** 由设计 I 型梁的横截面组成,以使梁在一定应力下不会变形或破坏的情况下获得最小体积。有关更多详细信息,请参阅 [Wikipedia](https://en.wikipedia.org/wiki/Cantilever)。我们记 L、H、h1、b1 和 b2 为梁的尺寸。

梁的体积由下式给出:

$V = f(H, h_1, b_1, b_2) = \left[2h_1b_1 + \left(H - 2h_1\right)b_2\right]L$


约束条件为:

- 梁根部处的最大弯曲应力,定义为:

  $\g_1(H, h_1, b_1, b_2) = \frac{P L H}{2I}$

- 梁尖端处的最大挠度,定义为:

  $\g_2(H, h_1, b_1, b_2) = \frac{P L^3}{3EI}$

### 学习要点

- 创建一个外部函数,返回多个值
- 在外部函数上启用代理建模
- 为该函数设置求值次数限制

注意:本示例旨在说明如何定义一个返回数组的外部函数,以及如何在简单问题上使用代理建模。由于本问题中使用的外部函数计算开销很小,因此不使用代理建模功能也可以求解。事实上,代理建模仅在目标函数的计算代价昂贵时才有用。在本示例中,我们仅出于演示目的使用它。


## 建模方法

悬臂梁问题的 OptAgent 模型使用整型和浮点型决策变量。H、b1、b2 声明为浮点型决策变量,定义域分别为 [3.0, 7.0]、[2.0, 12.0] 和 [0.1, 2.0]。最后一个变量 h1 是离散的:它表示索引集合 {0, 1, ..., 7} 中的整数,通过数组查表映射到一组可行取值 {0.1, 0.26, 0.35, 0.5, 0.65, 0.75, 0.9, 1.0}。

由于 OptAgent 的外部函数当前只支持返回单值,这里我们直接把弯曲应力、挠度和体积的公式内联到模型中,并对公式中的中间值 I 使用表达式节点。


## Python 实现


In [1]:
from optagent import ModelBuilder, solve

# Constant declaration
P = 1000
E = 10.0e6
L = 36
possible_values = [0.1, 0.26, 0.35, 0.5, 0.65, 0.75, 0.9, 1.0]

# Declare the optimization model
model = ModelBuilder()
possible_values_array = model.array(possible_values)

# Numerical decisions
H = model.float(default=5.0, lb=3.0, ub=7.0, name="H")
h1_index = model.int(default=0, lb=0, ub=len(possible_values) - 1, name="h1_index")
b1 = model.float(default=7.0, lb=2.0, ub=12.0, name="b1")
b2 = model.float(default=1.0, lb=0.1, ub=2.0, name="b2")
h1 = model.at(possible_values_array, h1_index)

# Moment of inertia and constraint/objective values
I = (1.0 / 12.0) * b2 * (H - 2 * h1) ** 3 + 2 * (
    (1.0 / 12.0) * b1 * h1 ** 3
    + b1 * h1 * (H - h1) ** 2 / 4.0
)
g1 = P * L * H / (2 * I)
# Use ** -1 to invert: OptAgent Exprs do not implement rtruediv for Python ints.
g2 = (P * L ** 3) * (3 * E * I) ** -1
volume = (2 * h1 * b1 + (H - 2 * h1) * b2) * L

# Constraints: bending stress and tip deflection
model.constraint(g1 <= 5000, name="bending_stress")
model.constraint(g2 <= 0.10, name="tip_deflection")

# Objective: minimize the beam volume
model.minimize(volume, name="volume")

# Solve and print the solution
solution = solve(model, time_limit_s=10.0)
print(f"status: {solution.status.value}")
print(f"objective = {solution.objective_value}")
H_value = solution.variable_values[H.node_id]
h1_index_value = solution.variable_values[h1_index.node_id]
b1_value = solution.variable_values[b1.node_id]
b2_value = solution.variable_values[b2.node_id]
print(f"H = {H_value}")
print(f"h1 = {possible_values[h1_index_value]}")
print(f"b1 = {b1_value}")
print(f"b2 = {b2_value}")


Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 96.0487
  improvements: initial=8 search=60
  evaluated: 9436
  wall_time: 10.003s
  termination: wall_time_exhausted


status: feasible
objective = 96.0487467961346
H = 6.993896922321133
h1 = 0.26
b1 = 3.88582894635569
b2 = 0.1
